# DepthForge — train & evaluate on NYU Depth V2 (Colab / Kaggle GPU)
Runtime → Change runtime type → **T4 GPU**. Total ≈ 35 min. Replace the repo URL with yours.

In [ ]:
!git clone https://github.com/<your-user>/depthforge.git
%cd depthforge
!pip -q install opencv-python-headless scikit-image h5py gradio

In [ ]:
# NYU v2 labeled set (2.8 GB, 1449 RGB-D pairs). Standard split: 0-999 train(900)/val(100), 1000-1448 test
!wget -q --show-progress -c http://horatio.cs.nyu.edu/mit/silberman/nyu_depth_v2/nyu_depth_v2_labeled.mat
# mirror if the NYU server is slow:
# !pip -q install gdown; # or use the Kaggle dataset 'nyu-depth-v2-labeled'

In [ ]:
import torch; print(torch.cuda.get_device_name(0))
!python scripts/train.py --data nyu --mat nyu_depth_v2_labeled.mat --minutes 60 --crop 128 --batch 32 --workers 2 --init weights/depthforge_net.pt

In [ ]:
# Submission evaluation on the 449 unseen NYU test images (reduce --n for a quick run)
!python scripts/evaluate.py --data nyu --mat nyu_depth_v2_labeled.mat --n 100 --team T01 --usns 'USN1,USN2,USN3,USN4'
!python scripts/quantize.py --n 10

In [ ]:
from google.colab import files
files.download('weights/depthforge_net.pt')
!zip -qr results.zip results && echo ok
files.download('results.zip')

Copy `weights/depthforge_net.pt` into the repo's `weights/` folder and redeploy (Docker / HF Space). Commit `results/eval_P4_T01.md`.